# VoxShield — run 2: voice conversion restored

Run 1 held attacks **A05 and A06** out of training. Those turned out to be the
**only two voice-conversion attacks** in ASVspoof 2019 LA train/dev — so the
model trained on text-to-speech only and never saw a single VC example.

In eval, **A17/A18/A19 are the voice-conversion attacks**, and that is exactly
where run 1 failed:

| family | attacks | run 1 mean miss rate |
|---|---|---|
| TTS | A07–A12, A16 | **0.02 %** |
| TTS/VC hybrid | A13–A15 | **0.00 %** |
| **VC** | **A17–A19** | **48.8 %** |

Overall: EER 11.27 %, ROC-AUC 0.950.

This notebook re-runs training on the **full** training set, which restores the
VC examples, and then compares the two runs attack by attack.

### Before you run anything

1. **Settings → Accelerator → GPU T4 ×2** (a P100 will not work — Kaggle's
   PyTorch dropped Pascal support)
2. **Settings → Internet → On**
3. **Add Data → `asvpoof-2019-dataset-la`**
4. **Add Data → Your Work → Notebook Output → your run-1 notebook**
   *(optional — only needed for the side-by-side comparison at the end)*

Then **Save Version → Save & Run All (Commit)** and close the laptop.
Roughly one hour.

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os, torch

print("\ntorch     ", torch.__version__)
print("cuda      ", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu {i}     {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}")
print("cpus      ", os.cpu_count())

major = torch.cuda.get_device_capability()[0] if torch.cuda.is_available() else 0
assert torch.cuda.is_available(), "No GPU. Settings > Accelerator > GPU T4 x2."
assert major >= 7, (
    f"sm_{major}x is too old for this PyTorch build (needs sm_70+). "
    "Switch Accelerator to GPU T4 x2 - a P100 is sm_60 and will not run."
)

# T4 is Turing: fp16 tensor cores, NO native bf16. is_bf16_supported() still
# returns True there via emulation, which is slower - so fp16 is forced below.
print("\nprecision : fp16 (forced - correct for T4)")

In [ ]:
import socket
try:
    socket.setdefaulttimeout(8)
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("pypi.org", 443))
    print("internet: ON")
except Exception:
    raise SystemExit(
        "internet: OFF - Settings > Internet > On (needs phone verification)."
    )

## 2 · Code and dependencies

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# Step out of the repo before deleting it - re-running this cell would
# otherwise remove the kernel's own working directory.
os.chdir("/kaggle/working")

WORK = Path("/kaggle/working/voxshield")
if WORK.exists():
    shutil.rmtree(WORK)

subprocess.run(
    ["git", "clone", "-q", "-b", "karthik", "https://github.com/SathvikGuttula/NullBox.git", str(WORK)], check=True
)
sys.path.insert(0, str(WORK / "backend"))

%cd /kaggle/working/voxshield
!git log --oneline -3

In [ ]:
!pip install -q "transformers>=4.44,<5" soundfile

import importlib
for m in ["torch", "torchaudio", "transformers", "soundfile", "librosa", "sklearn"]:
    try:
        print(f"{m:14}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:14} MISSING  {type(e).__name__}")

## 3 · Recover run 1's report (optional)

Only needed for the comparison in section 8. If you did not attach the run-1
notebook output, this skips and everything else still works.

In [ ]:
import glob, shutil
from pathlib import Path

RUN1 = Path("/kaggle/working/eval_heldout")

found = glob.glob("/kaggle/input/*/eval_la_eval/report.json")
found += glob.glob("/kaggle/input/*/*/eval_la_eval/report.json")

if found:
    RUN1.mkdir(parents=True, exist_ok=True)
    shutil.copy2(found[0], RUN1 / "report.json")
    import json
    r = json.loads((RUN1 / "report.json").read_text())
    print(f"run 1 recovered from {found[0]}")
    print(f"  EER {r['eer']*100:.2f}%   ROC-AUC {r['roc_auc']:.4f}")
else:
    print("run 1 output not attached - the comparison in section 8 will skip.")
    print("To enable it: Add Data > Your Work > Notebook Output > run-1 notebook")

## 4 · Manifests

Same as run 1 — the holdout files are still produced, but training will use the
**full** `train.csv` this time.

In [ ]:
!python backend/scripts/build_manifest.py \
    --discover /kaggle/input/datasets \
    --manifest-dir /kaggle/working/voxshield/datasets/manifests \
    --holdout-attacks A05,A06

In [ ]:
from pathlib import Path

# Fail here, before anything expensive, if the manifests are wrong.
for name, expected in [("train.csv", 25380), ("validation.csv", 24844),
                       ("test.csv", 71237)]:
    path = Path("datasets/manifests") / name
    assert path.exists(), f"missing {path} - did discovery find the dataset?"
    rows = sum(1 for _ in path.open(encoding="utf-8")) - 1
    status = "OK" if rows == expected else f"expected {expected}"
    print(f"  {name:<16} {rows:>7} rows   {status}")
    assert rows > 0, f"{name} is empty"

## 5 · Cache the full splits

About 9 minutes. `/kaggle/temp` is never saved between sessions, so this is
rebuilt each time — which is why the checkpoint, not the cache, goes to
`/kaggle/working`.

In [ ]:
!mkdir -p /kaggle/temp/cache

!python backend/scripts/cache_dataset.py \
    --manifest datasets/manifests/train.csv \
    --output /kaggle/temp/cache/train \
    --max-seconds 6.0

!python backend/scripts/cache_dataset.py \
    --manifest datasets/manifests/validation.csv \
    --output /kaggle/temp/cache/validation \
    --max-seconds 6.0

!du -sh /kaggle/temp/cache/* ; df -h /kaggle/temp | tail -1

## 6 · Train

Changes from run 1:

| | run 1 | run 2 |
|---|---|---|
| training set | `train_heldout.csv` (17,780, **no VC**) | `train.csv` (25,380, **VC included**) |
| validation | `val_unseen.csv` (A05/A06 only) | `validation.csv` (full dev) |
| unfrozen layers | 6 | **4** — run 1 hit 99.5 % train accuracy while validation worsened |
| precision | auto → bf16 (emulated on T4) | **fp16**, explicit |

`--resume` means a killed session costs one epoch, not the run.

In [ ]:
!python backend/scripts/train_model.py \
    --cache-dir /kaggle/temp/cache \
    --train-manifest datasets/manifests/train.csv \
    --validation-manifest datasets/manifests/validation.csv \
    --batch-size 32 --gradient-accumulation 1 \
    --precision fp16 \
    --epochs 5 --freeze-epochs 1 --unfreeze-top-layers 4 \
    --num-workers 2 \
    --experiment-id kaggle-full \
    --experiments-dir /kaggle/working/experiments \
    --model-out /kaggle/working/models/voxshield_full.pt \
    --resume

In [ ]:
from pathlib import Path

model = Path("/kaggle/working/models/voxshield_full.pt")
assert model.exists(), "TRAINING PRODUCED NO MODEL - check the cell above"
print(f"checkpoint OK  {model.stat().st_size/1024**2:.0f} MB")

## 7 · Evaluate on LA eval

71,237 utterances, attacks A07–A19, none seen in training.

No `--calibrate-on` this time: in run 1 it made calibration error *worse*
(0.168 → 0.221), because the fit came from a validation split whose attack mix
differs from test. It costs 4 minutes and does not affect EER or ROC-AUC, which
are invariant to a monotone temperature.

In [ ]:
!python backend/scripts/evaluate_model.py \
    --manifest datasets/manifests/test.csv \
    --model /kaggle/working/models/voxshield_full.pt \
    --batch-size 32 --num-workers 2 \
    --save-scores \
    --output-dir /kaggle/working/eval_full

## 8 · Compare the two runs

The per-attack table and the TTS-vs-VC summary are the actual result. A single
EER hides the whole story.

In [ ]:
from pathlib import Path
import subprocess

run1 = Path("/kaggle/working/eval_heldout/report.json")
run2 = Path("/kaggle/working/eval_full/report.json")

if run1.exists() and run2.exists():
    subprocess.run([
        "python", "backend/scripts/compare_runs.py",
        "--reports", str(run1), str(run2),
        "--names", "A05/A06 held out", "full train set",
        "--output", "/kaggle/working/comparison.md",
    ], cwd="/kaggle/working/voxshield", check=True)
elif run2.exists():
    print("Only run 2 is present - showing it alone.\n")
    subprocess.run([
        "python", "backend/scripts/compare_runs.py",
        "--reports", str(run2), "--names", "full train set",
        "--output", "/kaggle/working/comparison.md",
    ], cwd="/kaggle/working/voxshield", check=True)
else:
    print("No reports found - did section 7 run?")

## 9 · Summary and what to download

In [ ]:
import json
from pathlib import Path

W = Path("/kaggle/working")

print("=" * 62)
print("  RESULT")
print("=" * 62)

for name, path in [("run 1  (A05/A06 held out)", W / "eval_heldout/report.json"),
                   ("run 2  (full train set)  ", W / "eval_full/report.json")]:
    if path.exists():
        r = json.loads(path.read_text())
        per = r.get("per_attack", {})
        vc = [per[a]["miss_rate"] for a in ("A17", "A18", "A19") if a in per]
        line = (f"  {name}   EER {r['eer']*100:6.2f}%   "
                f"AUC {r['roc_auc']:.4f}   minDCF {r['min_dcf']:.4f}")
        if vc:
            line += f"   VC miss {sum(vc)/len(vc)*100:5.1f}%"
        print(line)
    else:
        print(f"  {name}   (not present)")

summary = W / "experiments/kaggle-full/summary.json"
if summary.exists():
    s = json.loads(summary.read_text())
    print(f"\n  best val EER {s['best_eer']*100:.3f}% at epoch {s['best_epoch']}"
          f"   ({s['epochs_run']} epochs run)")

print()
print("=" * 62)
print("  DOWNLOAD THESE  (Output tab, right-hand panel)")
print("=" * 62)

wanted = [
    ("models/voxshield_full.pt",                "the trained model - for YOU"),
    ("comparison.md",                           "run 1 vs run 2 table"),
    ("eval_full/report.json",                   "all metrics + per-attack"),
    ("eval_full/scores.csv",                    "per-utterance scores"),
    ("eval_full/test_det.png",                  "DET curve"),
    ("eval_full/test_roc.png",                  "ROC curve"),
    ("experiments/kaggle-full/summary.json",    "best EER + epoch"),
    ("experiments/kaggle-full/history.json",    "per-epoch curve"),
    ("experiments/kaggle-full/config.json",     "exact recipe"),
]

for relative, why in wanted:
    path = W / relative
    if path.exists():
        size = path.stat().st_size
        unit = f"{size/1024**2:7.1f} MB" if size > 1024**2 else f"{size/1024:7.1f} KB"
        print(f"  [x] {unit}  {relative:<38} {why}")
    else:
        print(f"  [ ] {'missing':>10}  {relative:<38} {why}")

print()
print("  The .pt file is for your own deployment and demo - keep it, but there")
print("  is no need to send it anywhere. Everything needed to review the result")
print("  is in the JSON and .md files, which total well under a megabyte.")
print()